In [1]:
!pip install azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 429.0/429.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 15.4 MB/s eta 0:00:00


In [2]:
!pip install -q PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 99.5 MB/s eta 0:00:00


In [3]:
!pip install -q PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.6 MB/s eta 0:00:00


In [12]:
import sys
if 'fitz' in sys.modules:
    del sys.modules['fitz']
if 'PyMuPDF' in sys.modules:
    del sys.modules['PyMuPDF']
!pip uninstall -y fitz PyMuPDF
!pip install -q PyMuPDF

Found existing installation: fitz 0.0.1.dev2
Uninstalling fitz-0.0.1.dev2:
  Successfully uninstalled fitz-0.0.1.dev2
Found existing installation: PyMuPDF 1.26.6
Uninstalling PyMuPDF-1.26.6:
  Successfully uninstalled PyMuPDF-1.26.6


In [6]:
from azure.storage.blob import BlobServiceClient
connection_string='DefaultEndpointsProtocol=https;AccountName=nareshitblob;AccountKey=hnclryr5KBlXTFUXdJhMeXXwS347QJ0Dthf6yFvUEBJ6foY5lVnGc07HqTcSEh5QclBfIEvKID7Q+AStDA4ctg==;EndpointSuffix=core.windows.net'
blob_service_client_using_connection_string=BlobServiceClient.from_connection_string(connection_string)
blob_service_client_using_connection_string

In [7]:
container_client=blob_service_client_using_connection_string \
                 .get_container_client("nareshitcontainer")
container_client

In [8]:
container_client=blob_service_client_using_connection_string \
                 .get_container_client("nareshitcontainer")
for blob in container_client.list_blobs():
    print(blob['name'])

Attention_all_you_need.pdf


In [9]:
blob_client=blob_service_client_using_connection_string \
            .get_blob_client("nareshitcontainer","Attention_all_you_need.pdf")

pdf_data=blob_client.download_blob()
pdf_bytes=pdf_data.readall()
len(pdf_bytes)

2215244

In [10]:
from PyPDF2 import PdfReader
import io
pdf_reader=PdfReader(io.BytesIO(pdf_bytes))
print(pdf_reader.pages[2].extract_text())

Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N= 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [ 11] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm( x+ Sublayer( x)), where Sublayer( x)is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512 .
Decoder: The decoder is also composed of a stack of N= 6identical layers.

In [13]:
import fitz  # PyMuPDF
import io

doc = fitz.open(stream=pdf_bytes, filetype="pdf")

for page_index in range(len(doc)):
    page = doc[page_index]
    image_list = page.get_images(full=True)

    for img_index, img in enumerate(image_list):
        xref = img[0]
        pix = fitz.Pixmap(doc, xref)

        if pix.alpha:  # handle alpha channel
            pix = fitz.Pixmap(fitz.csRGB, pix)

        pix.save(f"page{page_index}_img{img_index}.png")
